# Hybrid ARIMA-LSTM Forecast with Global Markets OHLCV

Demo notebook for:

| Slice | Dataset | Intervals |
|---|---|---|
| **Daily / weekly** | [benjaminpo/finance-dataset](https://www.kaggle.com/datasets/benjaminpo/finance-dataset) | `1d`, `1wk`, … |
| **Intraday** | [benjaminpo/finance-dataset-intraday](https://www.kaggle.com/datasets/benjaminpo/finance-dataset-intraday) | `1m` … `1h` (dated snapshots) |

It shows how to:
1. Locate both attached dataset roots on Kaggle (or a local `data/` checkout)
2. Load cumulative daily bars, plot **normalized prices**, and run a **hybrid ARIMA–LSTM** price forecast
3. Load all available dated intradaily snapshots and plot an intraday session

Pipeline / listings: [benjaminpo/finance-dataset](https://github.com/benjaminpo/finance-dataset)

## 1. Locate daily + intraday roots

On Kaggle the mounts are often nested as `/kaggle/input/datasets/<user>/<slug>/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.max_rows = 8
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

ASSET_MARKERS = (
    "stocks_us",
    "stocks_kr",
    "stocks_jp",
    "stocks_eu",
    "stocks_hk",
    "indices",
    "rates",
    "futures",
    "crypto",
    "currencies",
)
DAILY_INTERVALS = {"1d", "5d", "1wk", "1mo", "3mo"}
INTRADAY_INTERVALS = {"1m", "2m", "5m", "15m", "30m", "60m", "90m", "1h"}


def _looks_like_data_root(path: Path) -> bool:
    return path.is_dir() and any((path / asset).is_dir() for asset in ASSET_MARKERS)


def _iter_kaggle_candidates() -> list[Path]:
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.is_dir():
        print("/kaggle/input not present (local run)")
        return []

    mounted = sorted(p for p in kaggle_input.iterdir() if p.is_dir())
    print("Mounted under /kaggle/input:", [p.name for p in mounted] or "(none)")

    found: list[Path] = []
    frontier = list(mounted)
    for _ in range(4):
        nxt: list[Path] = []
        for root in frontier:
            if _looks_like_data_root(root):
                found.append(root)
                continue
            try:
                nxt.extend(sorted(p for p in root.iterdir() if p.is_dir()))
            except OSError:
                pass
        frontier = nxt
    return found


def _interval_dirs(root: Path) -> set[str]:
    names: set[str] = set()
    for asset in ASSET_MARKERS:
        asset_dir = root / asset
        if not asset_dir.is_dir():
            continue
        for child in asset_dir.iterdir():
            if child.is_dir():
                names.add(child.name)
    return names


def _score_daily(root: Path) -> int:
    intervals = _interval_dirs(root)
    score = len(intervals & DAILY_INTERVALS) * 10
    # Prefer the dedicated daily slug when both trees are present.
    name = root.name.lower()
    if "intraday" in name:
        score -= 50
    elif name.endswith("finance-dataset") or name == "finance-dataset":
        score += 20
    return score


def _score_intraday(root: Path) -> int:
    intervals = _interval_dirs(root)
    score = len(intervals & INTRADAY_INTERVALS) * 10
    path_l = str(root).lower()
    if "intraday" in path_l:
        score += 50
    if path_l.rstrip("/").endswith("finance-dataset-intraday"):
        score += 30
    return score


def _explicit_kaggle_roots() -> list[Path]:
    """Probe known mount layouts (Kaggle nests datasets inconsistently)."""
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.is_dir():
        return []
    slugs = ("finance-dataset", "finance-dataset-intraday")
    guesses: list[Path] = []
    for slug in slugs:
        guesses.extend(
            [
                kaggle_input / slug,
                kaggle_input / "datasets" / "benjaminpo" / slug,
                kaggle_input / "datasets" / slug,
            ]
        )
    return [p for p in guesses if _looks_like_data_root(p)]


def find_data_roots() -> tuple[Path, Path | None]:
    """Return (daily_root, intraday_root). Intraday may be None locally."""
    candidates = _iter_kaggle_candidates()
    candidates.extend(_explicit_kaggle_roots())

    here = Path.cwd()
    local = [
        here / "data",
        here.parent / "data",
        here.parent.parent / "data",
    ]
    candidates.extend(p for p in local if _looks_like_data_root(p))

    # Deduplicate while preserving order.
    uniq: list[Path] = []
    seen: set[Path] = set()
    for path in candidates:
        resolved = path.resolve() if path.exists() else path
        if resolved in seen or not _looks_like_data_root(path):
            continue
        seen.add(resolved)
        uniq.append(path)

    if not uniq:
        raise FileNotFoundError(
            "Could not find dataset root(s). Attach benjaminpo/finance-dataset and "
            "benjaminpo/finance-dataset-intraday on Kaggle, or use a local data/ dir."
        )

    daily = max(uniq, key=_score_daily)
    intraday_ranked = sorted(uniq, key=_score_intraday, reverse=True)
    intraday = intraday_ranked[0] if _score_intraday(intraday_ranked[0]) > 0 else None
    if intraday is not None and intraday.resolve() == daily.resolve():
        # Same tree (combined local checkout / legacy upload): reuse it.
        pass
    elif intraday is not None and "intraday" not in str(intraday).lower():
        # Only keep a non-named tree if it actually has intradaily intervals.
        if not (_interval_dirs(intraday) & INTRADAY_INTERVALS):
            intraday = None

    return daily, intraday


DAILY_DIR, INTRADAY_DIR = find_data_roots()
print("DAILY_DIR   =", DAILY_DIR)
print("INTRADAY_DIR=", INTRADAY_DIR)
if INTRADAY_DIR is None:
    print(
        "WARNING: intradaily root not found. In the notebook Input panel, attach "
        "benjaminpo/finance-dataset-intraday, then Restart session & re-run."
    )

## 2. Daily layout + load helpers

- **Cumulative** (`1d`, `1wk`, …): `{asset_class}/{interval}/{TICKER}.csv`
- Yahoo tickers like `^GSPC` / `EURUSD=X` become `GSPC.csv` / `EURUSD_X.csv` on disk.

In [ ]:
PREFERRED_DAILY = ("1d", "1wk", "1mo", "3mo", "5d")

asset_classes = sorted(
    p.name for p in DAILY_DIR.iterdir() if p.is_dir() and not p.name.startswith(".")
)
print("Daily asset classes:", ", ".join(asset_classes))

rows = []
for asset_class in asset_classes:
    asset_dir = DAILY_DIR / asset_class
    for name in PREFERRED_DAILY:
        interval_dir = asset_dir / name
        if not interval_dir.is_dir():
            continue
        sample = []
        for path in interval_dir.iterdir():
            if path.suffix == ".csv":
                sample.append(path.name)
                if len(sample) >= 3:
                    break
        if sample:
            rows.append(
                {
                    "asset_class": asset_class,
                    "interval": name,
                    "sample_files": ", ".join(sample),
                }
            )

pd.DataFrame(rows)

In [ ]:
def safe_filename(ticker: str) -> str:
    """Match the pipeline's on-disk naming (^GSPC → GSPC.csv, EURUSD=X → EURUSD_X.csv)."""
    return ticker.replace("^", "").replace("=", "_").replace("/", "_")


def load_ohlcv(
    asset_class: str,
    ticker: str,
    interval: str = "1d",
    *,
    data_dir: Path = None,
) -> pd.DataFrame:
    root = DAILY_DIR if data_dir is None else data_dir
    path = root / asset_class / interval / f"{safe_filename(ticker)}.csv"
    return pd.read_csv(path, parse_dates=["Datetime"], index_col="Datetime").sort_index()


def load_close(
    asset_class: str,
    ticker: str,
    interval: str = "1d",
    *,
    data_dir: Path = None,
) -> pd.Series:
    df = load_ohlcv(asset_class, ticker, interval=interval, data_dir=data_dir)
    col = "Adj Close" if "Adj Close" in df.columns else "Close"
    s = df[col].astype(float)
    s.name = ticker
    return s


aapl = load_ohlcv("stocks_us", "AAPL")
print(f"AAPL daily: {aapl.index.min().date()} → {aapl.index.max().date()} ({len(aapl)} bars)")
aapl.tail()

## 3. Daily multi-asset prices & returns

Normalized price (`P_t / P_0`) is the same curve as compounding daily simple returns from $1 — so we plot it once, then forecast with a hybrid ARIMA–LSTM in the next section.

In [ ]:
SERIES = [
    ("stocks_us", "AAPL"),
    ("stocks_us", "MSFT"),
    ("indices", "^GSPC"),
    ("crypto", "BTC-USD"),
    ("currencies", "EURUSD=X"),
]

closes = pd.concat(
    [load_close(asset_class, ticker) for asset_class, ticker in SERIES],
    axis=1,
).dropna(how="any")

LOOKBACK_DAYS = 365 * 3
window = closes.iloc[-LOOKBACK_DAYS:]
returns = window.pct_change().dropna(how="any")
# Same shape as window / window.iloc[0] (growth of $1 from the first bar).
normalized = window.div(window.iloc[0])

print(
    f"Window: {window.index.min().date()} → {window.index.max().date()} "
    f"({len(window)} bars)"
)
returns.describe().T[["mean", "std", "min", "max"]]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
normalized.plot(ax=ax, lw=1.4)
ax.set_title("Normalized price (start = 1.0) — same as growth of $1")
ax.set_ylabel("Index")
ax.set_xlabel("Datetime (UTC)")
ax.legend(loc="upper left", fontsize=9)
plt.show()

In [ ]:
corr = returns.corr()

fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Daily return correlation")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.show()
corr

## 4. Hybrid ARIMA–LSTM price forecast

Demo only — not investment advice.

**Architecture:** Instead of a tiny residual correction, we train **two independent models** and blend them:

1. **ARIMA** — rolling one-step log-price forecast (linear baseline)
2. **LSTM** — predicts next-day log-return from a multi-feature window (returns, moving averages, volatility, volume)

Final forecast = weighted average (optimized on a validation split):

$$\hat{y}_t = w \cdot \hat{y}^{\mathrm{ARIMA}}_t + (1-w) \cdot \hat{y}^{\mathrm{LSTM}}_t$$

This lets each model speak with its own voice instead of the LSTM trying to nudge a nearly-perfect random-walk forecast by fractions of a cent.


In [ ]:
import os
import warnings
from itertools import product

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import tensorflow as tf
from statsmodels.tsa.arima.model import ARIMA
from tensorflow import keras
from tensorflow.keras import layers

try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass

FORECAST_TICKER = "AAPL"
TEST_DAYS = 90
LOOKBACK = 30
EPOCHS = 80
BATCH_SIZE = 32
LSTM_UNITS = 64
SEED = 42

tf.keras.utils.set_random_seed(SEED)

price = window[FORECAST_TICKER].astype(float)
log_price = np.log(price)
log_ret = log_price.diff()

# Build multi-feature matrix: log-returns, rolling means, rolling vol, volume.
feat_df = pd.DataFrame(index=price.index)
feat_df["log_ret"] = log_ret
for w in [5, 10, 20]:
    feat_df[f"sma_{w}"] = log_ret.rolling(w).mean()
    feat_df[f"vol_{w}"] = log_ret.rolling(w).std()
try:
    ohlcv = load_ohlcv(
        next(a for a, t in SERIES if t == FORECAST_TICKER),
        FORECAST_TICKER,
    ).reindex(price.index)
    feat_df["volume_z"] = (
        np.log1p(ohlcv["Volume"].astype(float))
        .pipe(lambda s: (s - s.rolling(20).mean()) / s.rolling(20).std())
        .fillna(0.0)
    )
except Exception:
    feat_df["volume_z"] = 0.0

feat_df = feat_df.dropna()
# Target: next-day log-return.
target = log_ret.shift(-1).reindex(feat_df.index).dropna()
feat_df = feat_df.loc[target.index]

split = len(feat_df) - TEST_DAYS
val_start = int(split * 0.85)

# --- ARIMA on integer-indexed log-price ---
log_train_ri = pd.Series(log_price.iloc[: log_price.index.get_loc(feat_df.index[split]) + 1].to_numpy(dtype=float))
log_test_dates = feat_df.index[split:]


def _arima_fit(series, order):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit = ARIMA(series, order=order).fit(method_kwargs={"maxiter": 200})
    mle = getattr(fit, "mle_retvals", None) or {}
    if mle and not mle.get("converged", True):
        return None
    if not np.isfinite(getattr(fit, "aic", np.nan)):
        return None
    return fit


def select_arima_order(series, max_p=2, max_q=2):
    best_order, best_aic = (1, 1, 0), np.inf
    for p, q in product(range(max_p + 1), range(max_q + 1)):
        if p == 0 and q == 0:
            continue
        try:
            fit = _arima_fit(series, (p, 1, q))
            if fit is not None and fit.aic < best_aic:
                best_aic, best_order = fit.aic, (p, 1, q)
        except Exception:
            continue
    return best_order


order = select_arima_order(log_train_ri)
arima_res = _arima_fit(log_train_ri, order) or ARIMA(log_train_ri, order=(1, 1, 0)).fit()
print(f"{FORECAST_TICKER}: ARIMA{order} on train ({feat_df.index[0].date()} → {feat_df.index[split-1].date()})")

# Roll ARIMA through holdout.
arima_state = arima_res
arima_log_preds = []
next_ix = len(log_train_ri)

for t in range(TEST_DAYS):
    arima_hat = float(arima_state.forecast(steps=1).iloc[0])
    arima_log_preds.append(arima_hat)
    actual_log = float(log_price.loc[log_test_dates[t]])
    arima_state = arima_state.append(
        pd.Series([actual_log], index=[next_ix + t]), refit=False
    )

arima_pred_price = pd.Series(
    np.exp(arima_log_preds), index=log_test_dates, name="arima"
)

# --- LSTM on feature windows → next-day log-return ---
feat_arr = feat_df.to_numpy(dtype=np.float32)
tgt_arr = target.to_numpy(dtype=np.float32)

# Standardize features (fit on train only).
feat_mean = feat_arr[:split].mean(axis=0)
feat_std = feat_arr[:split].std(axis=0)
feat_std[feat_std < 1e-8] = 1.0
feat_arr = (feat_arr - feat_mean) / feat_std


def make_windows(x, y, lookback):
    xw, yw = [], []
    for i in range(lookback, len(x)):
        xw.append(x[i - lookback : i])
        yw.append(y[i])
    return np.asarray(xw, dtype=np.float32), np.asarray(yw, dtype=np.float32)


x_all, y_all = make_windows(feat_arr, tgt_arr, LOOKBACK)
actual_split = split - LOOKBACK
val_split = val_start - LOOKBACK

x_train, y_train = x_all[:val_split], y_all[:val_split]
x_val, y_val = x_all[val_split:actual_split], y_all[val_split:actual_split]
x_test, y_test = x_all[actual_split:], y_all[actual_split:]

n_features = x_train.shape[2]

model = keras.Sequential(
    [
        layers.Input(shape=(LOOKBACK, n_features)),
        layers.LSTM(LSTM_UNITS, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(LSTM_UNITS // 2),
        layers.Dense(LSTM_UNITS // 4, activation="relu"),
        layers.Dense(1),
    ]
)
model.compile(optimizer=keras.optimizers.Adam(5e-4), loss="mse")
hist = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    verbose=0, shuffle=False,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=12, restore_best_weights=True
        )
    ],
)

lstm_log_ret_pred = model.predict(x_test, verbose=0).ravel()
# LSTM predicts next-day log-return → convert to log-price level.
prev_log_prices = log_price.loc[log_test_dates].to_numpy(dtype=np.float64)
lstm_log_price_pred = prev_log_prices + lstm_log_ret_pred
lstm_pred_price = pd.Series(np.exp(lstm_log_price_pred), index=log_test_dates, name="lstm")

# --- Optimal blend weight on validation set ---
arima_val_preds = []
arima_val_state = arima_res
val_dates = feat_df.index[val_start:split]
val_next_ix = len(log_train_ri) - (split - val_start)
# Refit ARIMA up to val_start for clean val predictions.
log_val_ri = pd.Series(log_price.iloc[: log_price.index.get_loc(feat_df.index[val_start]) + 1].to_numpy(dtype=float))
arima_val_state = ARIMA(log_val_ri, order=order).fit(method_kwargs={"maxiter": 200})
for t in range(len(val_dates)):
    hat = float(arima_val_state.forecast(steps=1).iloc[0])
    arima_val_preds.append(hat)
    actual_val_log = float(log_price.loc[val_dates[t]])
    arima_val_state = arima_val_state.append(
        pd.Series([actual_val_log], index=[len(log_val_ri) + t]), refit=False
    )

lstm_val_ret = model.predict(x_val, verbose=0).ravel()
val_prev_log = log_price.loc[val_dates].to_numpy(dtype=np.float64)
lstm_val_log = val_prev_log + lstm_val_ret

arima_val_arr = np.asarray(arima_val_preds)
actual_val_arr = log_price.shift(-1).loc[val_dates].to_numpy(dtype=np.float64)

best_w, best_mse = 0.5, np.inf
for w_try in np.linspace(0.0, 1.0, 101):
    blend = w_try * arima_val_arr + (1 - w_try) * lstm_val_log
    mse = float(np.mean((blend - actual_val_arr) ** 2))
    if mse < best_mse:
        best_mse, best_w = mse, w_try

print(f"Blend weight: {best_w:.2f} ARIMA + {1 - best_w:.2f} LSTM (val MSE={best_mse:.6f})")

# --- Blended test predictions ---
blended_log = best_w * np.asarray(arima_log_preds) + (1.0 - best_w) * np.asarray(lstm_log_price_pred)
predicted = pd.Series(np.exp(blended_log), index=log_test_dates, name="hybrid")

actual = price.shift(-1).reindex(log_test_dates).rename("actual")
# For alignment: we predicted for date t the close on t+1.
# Actually simpler: arima_log_preds[t] is one-step forecast of log_price at test_date[t]+1.
# lstm similarly. Let's just compare to next-day actual.
# Align actual to the date the forecast was *for* (next trading day).
actual_next = price.iloc[split + 1 : split + 1 + TEST_DAYS]
if len(actual_next) < TEST_DAYS:
    actual_next = price.iloc[split + 1 :]
actual_next = actual_next.iloc[: len(log_test_dates)]
actual = actual_next.rename("actual")
# Reindex predictions to match actual dates.
arima_pred_price = pd.Series(arima_pred_price.values[: len(actual)], index=actual.index, name="arima")
lstm_pred_price = pd.Series(lstm_pred_price.values[: len(actual)], index=actual.index, name="lstm")
predicted = pd.Series(predicted.values[: len(actual)], index=actual.index, name="hybrid")

err = predicted - actual
arima_err = arima_pred_price - actual
lstm_err = lstm_pred_price - actual

prev_close = price.shift(1).loc[actual.index]
actual_ret = (actual / prev_close - 1.0).rename("actual")
arima_ret = (arima_pred_price / prev_close - 1.0).rename("arima")
lstm_ret = (lstm_pred_price / prev_close - 1.0).rename("lstm")
hybrid_ret = (predicted / prev_close - 1.0).rename("hybrid")


def _metrics(pred):
    e = pred - actual
    return {
        "MAE": float(e.abs().mean()),
        "RMSE": float(np.sqrt((e**2).mean())),
        "MAPE%": float((e.abs() / actual).mean() * 100),
    }


naive = price.shift(1).loc[actual.index]
metrics = pd.DataFrame(
    {
        "naive": _metrics(naive),
        "ARIMA only": _metrics(arima_pred_price),
        "LSTM only": _metrics(lstm_pred_price),
        "hybrid": _metrics(predicted),
    }
).T

print(
    f"Holdout: {actual.index.min().date()} → {actual.index.max().date()} "
    f"({len(actual)} days) | LSTM epochs≈{len(hist.history['loss'])}, lookback={LOOKBACK}"
)
metrics


In [ ]:
ZOOM_DAYS = 30

# When the blend collapses to one model, hybrid overlays that line and hides it.
show_hybrid = 0.05 < best_w < 0.95
gap = (lstm_pred_price - arima_pred_price).abs()
print(
    f"Blend: {best_w:.0%} ARIMA + {1 - best_w:.0%} LSTM | "
    f"mean |LSTM − ARIMA| = ${gap.mean():.3f} (max ${gap.max():.3f})"
)
if not show_hybrid:
    winner = "LSTM" if best_w < 0.5 else "ARIMA"
    print(f"Hybrid ≡ {winner} at this weight — plotting ARIMA + LSTM only (no overlay).")

fig, axes = plt.subplots(4, 1, figsize=(11, 13), constrained_layout=True)


def _price_panel(ax, idx=None, title=""):
    a = actual if idx is None else actual.loc[idx]
    ar = arima_pred_price if idx is None else arima_pred_price.loc[idx]
    ls = lstm_pred_price if idx is None else lstm_pred_price.loc[idx]
    hy = predicted if idx is None else predicted.loc[idx]

    ax.plot(a.index, a.values, lw=2.2, color="0.2", label="Actual", zorder=1)
    # Solid thick LSTM drawn last among forecasts so it never disappears under hybrid.
    ax.plot(ar.index, ar.values, lw=1.6, ls="--", color="#ff7f0e", label="ARIMA", zorder=2)
    ax.plot(ls.index, ls.values, lw=2.0, color="#9467bd", label="LSTM", zorder=4)
    if show_hybrid:
        ax.plot(
            hy.index,
            hy.values,
            lw=1.5,
            ls="-.",
            color="#2ca02c",
            label=f"Hybrid ({best_w:.0%} A + {1 - best_w:.0%} L)",
            zorder=3,
        )
    ax.set_title(title)
    ax.set_ylabel("Price")
    ax.legend(loc="upper left", fontsize=9)


_price_panel(
    axes[0],
    title=f"{FORECAST_TICKER} — ARIMA{order} vs LSTM ({TEST_DAYS}-day holdout)",
)

z = actual.index[-ZOOM_DAYS:]
_price_panel(axes[1], idx=z, title=f"Zoom: last {ZOOM_DAYS} days")

# Return-space: LSTM solid, ARIMA dashed — clearest place to see they differ.
axes[2].plot(
    actual_ret.index, actual_ret.values * 100, lw=1.0, color="0.45", alpha=0.55, label="Actual ret"
)
axes[2].plot(
    arima_ret.index, arima_ret.values * 100, lw=1.4, ls="--", color="#ff7f0e", label="ARIMA ret"
)
axes[2].plot(
    lstm_ret.index, lstm_ret.values * 100, lw=1.6, color="#9467bd", label="LSTM ret"
)
if show_hybrid:
    axes[2].plot(
        hybrid_ret.index,
        hybrid_ret.values * 100,
        lw=1.3,
        ls="-.",
        color="#2ca02c",
        label="Hybrid ret",
    )
axes[2].axhline(0, color="k", lw=0.7, alpha=0.4)
axes[2].set_title("One-step return forecasts (%) — LSTM (purple) vs ARIMA (orange dashed)")
axes[2].set_ylabel("Return %")
axes[2].legend(loc="upper left", fontsize=8, ncol=2)

axes[3].plot(
    arima_err.index,
    arima_err.abs().rolling(5).mean().values,
    lw=1.5,
    ls="--",
    color="#ff7f0e",
    label="|ARIMA err| 5d MA",
)
axes[3].plot(
    lstm_err.index,
    lstm_err.abs().rolling(5).mean().values,
    lw=1.8,
    color="#9467bd",
    label="|LSTM err| 5d MA",
)
if show_hybrid:
    axes[3].plot(
        err.index,
        err.abs().rolling(5).mean().values,
        lw=1.4,
        ls="-.",
        color="#2ca02c",
        label="|Hybrid err| 5d MA",
    )
axes[3].set_title("Absolute error (5-day moving average)")
axes[3].set_ylabel("Error ($)")
axes[3].set_xlabel("Datetime (UTC)")
axes[3].legend(loc="upper left", fontsize=9)

plt.show()
metrics


## 5. Intraday snapshots

Yahoo only keeps a short rolling window, so intradaily bars are stored as **dated files**:

`{asset_class}/{interval}/{TICKER}_{YYYY-MM-DD}.csv`

Example: `stocks_us/5m/AAPL_2026-07-21.csv`. Older snapshots accumulate across runs — load **every** dated file for a ticker to rebuild the longest available series.

In [ ]:
if INTRADAY_DIR is None:
    kaggle_input = Path("/kaggle/input")
    mounted = (
        sorted(p.name for p in kaggle_input.iterdir() if p.is_dir())
        if kaggle_input.is_dir()
        else []
    )
    raise FileNotFoundError(
        "Intraday dataset not found. Attach benjaminpo/finance-dataset-intraday "
        "in the notebook Input panel (then Restart session & re-run), or use a "
        "local data/ tree that still contains 1m…1h folders. "
        f"Currently mounted under /kaggle/input: {mounted or '(none)'}"
    )


def snapshot_path(
    asset_class: str,
    ticker: str,
    day: str,
    interval: str = "5m",
    *,
    data_dir: Path | None = None,
) -> Path:
    root = INTRADAY_DIR if data_dir is None else data_dir
    return root / asset_class / interval / f"{safe_filename(ticker)}_{day}.csv"


def available_snapshot_days(
    asset_class: str,
    ticker: str,
    interval: str = "5m",
    *,
    lookback_calendar_days: int = 400,
    data_dir: Path | None = None,
) -> list[str]:
    """Probe calendar dates for snapshot files (avoids listing huge interval dirs)."""
    days: list[str] = []
    today = pd.Timestamp.utcnow().normalize()
    for offset in range(lookback_calendar_days):
        day = (today - pd.Timedelta(days=offset)).strftime("%Y-%m-%d")
        if snapshot_path(asset_class, ticker, day, interval, data_dir=data_dir).is_file():
            days.append(day)
    return sorted(days)


def load_intraday(
    asset_class: str,
    ticker: str,
    interval: str = "5m",
    *,
    days: list[str] | None = None,
    last_n_days: int | None = None,
    lookback_calendar_days: int = 400,
    data_dir: Path | None = None,
) -> pd.DataFrame:
    """Load and concatenate dated intradaily snapshot CSVs (all found days by default)."""
    root = INTRADAY_DIR if data_dir is None else data_dir
    available = days or available_snapshot_days(
        asset_class,
        ticker,
        interval,
        lookback_calendar_days=lookback_calendar_days,
        data_dir=root,
    )
    if not available:
        raise FileNotFoundError(
            f"No {interval} snapshots for {ticker} under {root / asset_class / interval} "
            f"in the last {lookback_calendar_days} calendar days"
        )

    if last_n_days is not None:
        available = available[-last_n_days:]

    frames = [
        pd.read_csv(
            snapshot_path(asset_class, ticker, day, interval, data_dir=root),
            parse_dates=["Datetime"],
            index_col="Datetime",
        )
        for day in available
    ]
    df = pd.concat(frames).sort_index()
    return df[~df.index.duplicated(keep="last")]


# Prefer US equity 5m; fall back to BTC if needed.
INTRADAY_CANDIDATES = [
    ("stocks_us", "AAPL", "5m"),
    ("stocks_us", "MSFT", "5m"),
    ("crypto", "BTC-USD", "5m"),
    ("crypto", "BTC-USD", "15m"),
]

intraday_meta = None
for asset_class, ticker, interval in INTRADAY_CANDIDATES:
    days = available_snapshot_days(asset_class, ticker, interval)
    if days:
        intraday_meta = (asset_class, ticker, interval, days)
        break

if intraday_meta is None:
    raise FileNotFoundError(
        f"No intradaily snapshots under {INTRADAY_DIR} for {INTRADAY_CANDIDATES}"
    )

asset_class, ticker, interval, found_days = intraday_meta
print(
    f"Using {asset_class}/{interval}/{safe_filename(ticker)}_*.csv — "
    f"found {len(found_days)} day(s), "
    f"{found_days[0]} → {found_days[-1]}"
)
print("Days:", ", ".join(found_days))

# Load every available snapshot day (pass last_n_days=N to cap).
intraday = load_intraday(asset_class, ticker, interval, days=found_days)
print(
    f"Loaded {len(intraday)} bars across {len(found_days)} session(s): "
    f"{intraday.index.min()} → {intraday.index.max()}"
)
intraday.tail()


In [ ]:
price_col = "Adj Close" if "Adj Close" in intraday.columns else "Close"
latest_day = str(intraday.index.max().date())
session = intraday.loc[latest_day]

fig, axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)

intraday[price_col].plot(ax=axes[0], lw=1.2, color="C0")
axes[0].set_title(f"{ticker} {interval} — last {intraday.index.normalize().nunique()} session(s)")
axes[0].set_ylabel(price_col)
axes[0].set_xlabel("")

session[price_col].plot(ax=axes[1], lw=1.4, color="C1")
axes[1].set_title(f"{ticker} {interval} — {latest_day} only")
axes[1].set_ylabel(price_col)
axes[1].set_xlabel("Datetime (UTC)")

plt.show()

# Simple intradaily return stats for the loaded window.
intra_ret = intraday[price_col].pct_change().dropna()
pd.Series(
    {
        "bars": len(intraday),
        "sessions": int(intraday.index.normalize().nunique()),
        "mean_ret": float(intra_ret.mean()),
        "std_ret": float(intra_ret.std()),
        "min_ret": float(intra_ret.min()),
        "max_ret": float(intra_ret.max()),
    },
    name=f"{ticker} {interval}",
).to_frame("value")

## 6. Path cheat sheet

| Example | Dataset | Path |
|---|---|---|
| Apple daily | daily | `stocks_us/1d/AAPL.csv` |
| Apple weekly | daily | `stocks_us/1wk/AAPL.csv` |
| Bitcoin daily | daily | `crypto/1d/BTC-USD.csv` |
| S&P 500 | daily | `indices/1d/GSPC.csv` (`^` stripped) |
| EUR/USD | daily | `currencies/1d/EURUSD_X.csv` (`=` → `_`) |
| AAPL 5-minute day | intraday | `stocks_us/5m/AAPL_YYYY-MM-DD.csv` |
| BTC 1-minute day | intraday | `crypto/1m/BTC-USD_YYYY-MM-DD.csv` |

Asset classes: `stocks_us`, `stocks_kr`, `stocks_jp`, `stocks_eu`, `stocks_hk`, `indices`, `rates`, `futures`, `crypto`, `currencies`.